In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("WeaveIQ environment is ready!")

WeaveIQ environment is ready!


In [3]:
df = pd.read_csv("../data/raw/full_weaving_dataset.csv", low_memory=False)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 121148
Columns: 18


,ID,Month,Construction,Req_Finish_Fabrics,Fabric_Allowance,Rec_Beam_length(yds),Shrink_allow,act_shrink%,Previous_pdn,Req_grey_fabric,Req_beam_length(yds),Total_pdn_m/c,Rej_and_cut_Piece,Total_pdn_per_order,warp_count,weft_count,epi,ppi
0,12207-8,January,40+40/2/40/110x80,31300,6.0,5752.336,12.5,12.26173158,5047,33297.87234,34797.6511,5047.0,0,0,double,80,110,80
1,12207-8,January,40+40/2/40/110x80,31300,6.0,5883.568,12.5,64.12041129,1952,33297.87234,34797.6511,2111.0,0,0,double,80,110,80
2,12207-8,January,40+40/2/40/110x80,31300,6.0,3094.888,12.5,24.13295732,2207,33297.87234,34797.6511,2348.0,0,0,double,80,110,80
3,12207-8,January,40+40/2/40/110x80,31300,6.0,5894.504,12.5,14.734132,5026,33297.87234,34797.6511,5026.0,0,0,double,80,110,80
4,12207-8,January,40+40/2/40/110x80,31300,6.0,5850.760,12.5,21.46319453,4391,33297.87234,34797.6511,4595.0,0,0,double,80,110,80


In [4]:
column_summary = pd.DataFrame({
    "Data type": df.dtypes,
    "Missing values": df.isna().sum(),
    "Unique values": df.nunique()
})

display(column_summary)

,Data type,Missing values,Unique values
ID,str,0,1009
Month,str,0,9
Construction,str,0,128
Req_Finish_Fabrics,int64,0,379
Fabric_Allowance,float64,0,106
Rec_Beam_length(yds),float64,0,1495
Shrink_allow,float64,0,110
act_shrink%,str,0,27443
Previous_pdn,str,13637,3304
Req_grey_fabric,float64,0,658


In [5]:
# Identify rows marked TOTAL without changing the original data
is_total = (
    df["Previous_pdn"]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq("TOTAL")
    .fillna(False)
)

print("TOTAL summary rows:", is_total.sum())
print("Other rows:", (~is_total).sum())
print("Unique IDs in TOTAL rows:", df.loc[is_total, "ID"].nunique())
print("Exact duplicate rows beyond the first:", df.duplicated().sum())

# Compare example summary records
display(
    df.loc[is_total, [
        "ID",
        "Month",
        "Previous_pdn",
        "Rej_and_cut_Piece",
        "Total_pdn_per_order"
    ]].head(10)
)

TOTAL summary rows: 22022
Other rows: 99126
Unique IDs in TOTAL rows: 978
Exact duplicate rows beyond the first: 74448


,ID,Month,Previous_pdn,Rej_and_cut_Piece,Total_pdn_per_order
7,12207-8,January,TOTAL,285,27982
15,12228-1,January,TOTAL,39,10659
18,12249-6(B),January,TOTAL,0,1389
23,12277-5,January,TOTAL,58,8795
58,12284-1(A),January,TOTAL,1043,61001
61,12285-4,January,TOTAL,0,1326
70,12320-1(A),January,TOTAL,203,10997
74,12301-1(A),January,TOTAL,73,2290
77,12305-1(A),January,TOTAL,49,2052
209,12324-3,January,TOTAL,858,249732


In [6]:
# IDs that have at least one TOTAL summary row
summary_ids = df.loc[is_total, "ID"].unique()

# Records belonging to orders with no TOTAL summary row
no_summary = df.loc[~df["ID"].isin(summary_ids)]

print("Orders without TOTAL rows:", no_summary["ID"].nunique())

# Positive rejection amounts recorded outside TOTAL rows
positive_other = df.loc[
    (~is_total) & (df["Rej_and_cut_Piece"] > 0)
]

print("Non-TOTAL rows with positive rejection:", len(positive_other))
print("Orders involved:", positive_other["ID"].nunique())

display(
    positive_other[[
        "ID",
        "Month",
        "Previous_pdn",
        "Rej_and_cut_Piece",
        "Total_pdn_per_order"
    ]].head(10)
)

Orders without TOTAL rows: 31
Non-TOTAL rows with positive rejection: 161
Orders involved: 10


,ID,Month,Previous_pdn,Rej_and_cut_Piece,Total_pdn_per_order
2388,SF-11037,January,785572,4642,788898
5476,SF-11037,January,890717,5151,895927
6113,SF-11037,January,915199,5326,924098
13514,SF-11037,January,884501,5504,889479
16617,SF-11037,January,872477,5389,876106
18623,SF-11037,February,527728,3204,521723
19001,SF-11037,February,540796,3292,543750
20610,SF-11037,February,620338,3774,619413
29233,SF-11037,February,748148,5177,748609
29749,SF-11037,February,748148,5177,748609


In [7]:
# Summarize IDs with positive rejection outside TOTAL rows
other_summary = positive_other.groupby("ID").agg(
    positive_rows=("Rej_and_cut_Piece", "size"),
    months_present=("Month", "nunique")
)

# Check whether each ID also appears in TOTAL rows
other_summary["has_TOTAL_row"] = other_summary.index.isin(summary_ids)

display(other_summary)

,positive_rows,months_present,has_TOTAL_row
ID,,,
13178-2,21,1,True
SF-11037,32,3,True
SF-12263,56,2,True
SF-13151,12,1,True
SF-13176,5,1,True
SF-13189,3,1,True
SF-13214,3,1,True
SF-13249,2,1,True
SF-13254,5,1,True


In [8]:
# Inspect one order while preserving its original file order
order_check = df.loc[
    df["ID"] == "SF-11037",
    [
        "Month",
        "Previous_pdn",
        "Rej_and_cut_Piece",
        "Total_pdn_per_order"
    ]
].copy()

order_check["is_TOTAL"] = is_total.loc[order_check.index]

print("Records for this ID:", len(order_check))

# Show the last 20 records in file order
display(order_check.tail(20))

Records for this ID: 633


,Month,Previous_pdn,Rej_and_cut_Piece,Total_pdn_per_order,is_TOTAL
40538,March,NaN,0,0,False
40539,March,NaN,0,0,False
40965,March,127,0,0,False
40966,March,NaN,0,0,False
40967,March,TOTAL,0,0,True
40968,March,NaN,0,0,False
40969,March,406738,3367,402374,False
40970,March,NaN,0,0,False
40971,March,NaN,0,0,False
40972,March,NaN,0,0,False


In [9]:
# Last recorded row for each ID, preserving file order
last_rows = df.drop_duplicates(subset="ID", keep="last").copy()

# IDs with positive production somewhere in their history
produced_ids = df.loc[
    df["Total_pdn_per_order"] > 0, "ID"
].unique()

# Last row has zero production despite earlier positive production
suspicious_last = last_rows.loc[
    last_rows["Total_pdn_per_order"].eq(0)
    & last_rows["ID"].isin(produced_ids)
]

print("Unique IDs:", len(last_rows))
print("IDs ending in zero production after positive production:",
      len(suspicious_last))

display(
    suspicious_last[[
        "ID", "Month", "Previous_pdn",
        "Rej_and_cut_Piece", "Total_pdn_per_order"
    ]].head(10)
)

Unique IDs: 1009
IDs ending in zero production after positive production: 98


,ID,Month,Previous_pdn,Rej_and_cut_Piece,Total_pdn_per_order
17361,13001-3,January,3042,0,0
17433,13006-2,January,1915,0,0
17459,13009-1,January,2332,0,0
17460,13022-1,January,2434,0,0
40111,13056-2,March,2058,0,0
40547,13056-3,March,1987,0,0
40573,13061-2,March,2079,0,0
40604,13064-6,March,2169,0,0
46949,13064-5,April,2721,0,0
49319,13068-2,April,2332,0,0


In [10]:
# Last TOTAL row for each ID that has summary records
last_total = df.loc[is_total].drop_duplicates(
    subset="ID", keep="last"
).copy()

# Summary rows ending in zero despite positive production elsewhere
suspicious_total = last_total.loc[
    last_total["Total_pdn_per_order"].eq(0)
    & last_total["ID"].isin(produced_ids)
]

print("IDs with TOTAL rows:", len(last_total))
print("Last TOTAL row has zero after positive production:",
      len(suspicious_total))

display(
    suspicious_total[[
        "ID", "Month", "Rej_and_cut_Piece", "Total_pdn_per_order"
    ]].head(10)
)

IDs with TOTAL rows: 978
Last TOTAL row has zero after positive production: 16


,ID,Month,Rej_and_cut_Piece,Total_pdn_per_order
52893,SF-13001,April,0,0
54084,13072-9,May,0,0
65390,13088-1,May,0,0
69424,13147-2,June,0,0
109522,13013-21(A),September,0,0
109525,13014-3(A),September,0,0
109746,13042-1,September,0,0
109817,13050-1,September,0,0
109820,13050-2,September,0,0
109828,13050-3,September,0,0


In [11]:
# Collect all records belonging to the 16 flagged IDs
flagged_ids = suspicious_total["ID"]

review_rows = df.loc[df["ID"].isin(flagged_ids)].copy()

# Preserve each record's original position in the dataset
review_rows.insert(0, "source_row", review_rows.index)

review_rows.to_csv(
    "../reports/final_snapshot_review.csv",
    index=False
)

print("Saved: reports/final_snapshot_review.csv")

Saved: reports/final_snapshot_review.csv


In [12]:
# Find records for IDs that have no TOTAL row
summary_ids = df.loc[is_total, "ID"].unique()
no_summary = df.loc[~df["ID"].isin(summary_ids)]

# Summarize every one of these IDs
no_summary_check = no_summary.groupby("ID").agg(
    row_count=("ID", "size"),
    months_present=("Month", "nunique"),
    positive_production_rows=(
        "Total_pdn_per_order", lambda values: (values > 0).sum()
    ),
    positive_target_rows=(
        "Rej_and_cut_Piece", lambda values: (values > 0).sum()
    ),
    zero_target_rows=(
        "Rej_and_cut_Piece", lambda values: (values == 0).sum()
    )
)

print("IDs without TOTAL rows:", len(no_summary_check))
display(no_summary_check)

IDs without TOTAL rows: 31


,row_count,months_present,positive_production_rows,positive_target_rows,zero_target_rows
ID,,,,,
12284-1,3,1,0,0,3
12286-2,25,1,0,0,25
12301-1,16,1,0,0,16
12305-1,29,1,0,0,29
12307-2,15,1,0,0,15
12307-3,15,1,0,0,15
12308-1,22,1,0,0,22
12308-2,10,1,0,0,10
12313-1,11,1,0,0,11


In [13]:
no_summary_check.to_csv(
    "../reports/orders_without_total_audit.csv",
    index=True
)

print("Saved audit for all 31 IDs.")

Saved audit for all 31 IDs.


In [14]:
excel_path = "../data/raw/raw_data_all.xlsx"

workbook = pd.ExcelFile(excel_path, engine="openpyxl")
print("Sheet names:", workbook.sheet_names)

preview = pd.read_excel(
    workbook,
    sheet_name=0,
    header=None,
    nrows=12
)

with pd.option_context("display.max_columns", None):
    display(preview)

Sheet names: ['Sheet1']


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21
0,ID #,Construction,Order Quantity,NaN,NaN,Shrin.%,B.L.(m),L #,Rec.,Rec.,TODAYS,Previous,TOTAL,Shrin.%,Grey(Inspected)Production,NaN,NaN,NaN,NaN,Shrin.%,B/F,Remarks
1,NaN,NaN,F.F.,F.A,G.F.,NaN,NaN,NaN,B/L,B/L,Prodn.,Prodn.,Prodn.,NaN,Todays,Pre.,TOTAL,Rej.&,TOTAL,NaN,B/C,NaN
2,NaN,NaN,Yds,NaN,Yds,NaN,NaN,NaN,mtr,Yds,NaN,NaN,Yds,NaN,Del.,Del.,Del.,C.Pcs,Prodn.,NaN,D/C,NaN
3,12207-8,40+40/2/40/110x80,31300,6,33297.87234,12.5,34797.651103,104,5260,5752.336,0,5047,5047,12.261732,NaN,NaN,NaN,NaN,NaN,NaN,B/F,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102,5380,5883.568,159,1952,2111,64.120411,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,103,2830,3094.888,141,2207,2348,24.132957,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82,5390,5894.504,0,5026,5026,14.734132,NaN,NaN,NaN,NaN,NaN,NaN,B/F,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,141,5350,5850.76,204,4391,4595,21.463195,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,143,5400,5905.44,225,4340,4565,22.698393,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,144,5400,5905.44,200,3754,3954,33.044786,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Skip the three header rows; keep original blanks
raw_excel = pd.read_excel(
    excel_path,
    sheet_name="Sheet1",
    header=None,
    skiprows=3,
    engine="openpyxl"
)

print("Excel data rows:", len(raw_excel))
print("CSV data rows:", len(df))

if len(raw_excel) == len(df):
    # Column positions identified from the Excel headers
    comparisons = [
        ("Received beam length", 9, "Rec_Beam_length(yds)", "forward"),
        ("Rejected + cut pieces", 17, "Rej_and_cut_Piece", "zero"),
        ("Total order production", 18, "Total_pdn_per_order", "zero")
    ]

    for label, position, csv_column, treatment in comparisons:
        original = pd.to_numeric(
            raw_excel.iloc[:, position], errors="coerce"
        )

        # Reproduce the author's filling only for this comparison
        expected = (
            original.ffill()
            if treatment == "forward"
            else original.fillna(0)
        )

        actual = pd.to_numeric(df[csv_column], errors="coerce")

        matches = np.isclose(
            expected.to_numpy(dtype=float),
            actual.to_numpy(dtype=float),
            rtol=1e-9,
            atol=1e-6,
            equal_nan=True
        )

        print(f"{label}: {(~matches).sum()} mismatching rows")
else:
    print("Row counts differ. Do not transfer values between files yet.")

Excel data rows: 121148
CSV data rows: 121148
Received beam length: 0 mismatching rows
Rejected + cut pieces: 0 mismatching rows
Total order production: 0 mismatching rows


In [18]:
# Read the original target without filling missing values
original_target = raw_excel.iloc[:, 17]
numeric_target = pd.to_numeric(original_target, errors="coerce")

target_check = pd.Series({
    "Originally missing": original_target.isna().sum(),
    "Explicitly recorded zero": numeric_target.eq(0).sum(),
    "Positive amount": numeric_target.gt(0).sum(),
    "Negative amount": numeric_target.lt(0).sum(),
    "Non-numeric, non-missing": (
        original_target.notna() & numeric_target.isna()
    ).sum()
})

display(target_check)

Originally missing          98973
Explicitly recorded zero     6748
Positive amount             15427
Negative amount                 0
Non-numeric, non-missing        0
dtype: int64

In [19]:
# Compare original target availability by row type
target_audit = pd.DataFrame({
    "ID": df["ID"].to_numpy(),
    "row_type": np.where(is_total, "TOTAL", "Other"),
    "target_recorded": numeric_target.notna().to_numpy()
})

availability = pd.crosstab(
    target_audit["row_type"],
    target_audit["target_recorded"]
).rename(columns={False: "Missing target", True: "Recorded target"})

display(availability)

print(
    "IDs with at least one recorded target:",
    target_audit.loc[target_audit["target_recorded"], "ID"].nunique()
)

target_recorded,Missing target,Recorded target
row_type,,
Other,98965,161
TOTAL,8,22014


IDs with at least one recorded target: 978


In [20]:
# Preserve the complete dataset and restore original missing targets
audit_df = df.copy()
audit_df["Rej_and_cut_Piece"] = numeric_target.to_numpy()
audit_df["source_row"] = np.arange(len(audit_df))

# Keep only rows whose target was explicitly recorded
labeled_candidates = audit_df.loc[
    audit_df["Rej_and_cut_Piece"].notna()
].copy()

print("Candidate labeled rows:", len(labeled_candidates))
print("Unique IDs:", labeled_candidates["ID"].nunique())
print(
    "Recorded zero targets:",
    labeled_candidates["Rej_and_cut_Piece"].eq(0).sum()
)

Candidate labeled rows: 22175
Unique IDs: 978
Recorded zero targets: 6748


In [21]:
# Recover original production values without filling blanks
original_production = pd.to_numeric(
    raw_excel.iloc[:, 18], errors="coerce"
)

labeled_candidates["original_production"] = (
    original_production.to_numpy()[
        labeled_candidates["source_row"].to_numpy()
    ]
)

production = labeled_candidates["original_production"]
target = labeled_candidates["Rej_and_cut_Piece"]

checks = pd.Series({
    "Missing production": production.isna().sum(),
    "Zero production": production.eq(0).sum(),
    "Positive production": production.gt(0).sum(),
    "Negative production": production.lt(0).sum(),
    "Positive target with zero production": (
        target.gt(0) & production.eq(0)
    ).sum(),
    "Target greater than positive production": (
        production.gt(0) & target.gt(production)
    ).sum()
})

display(checks)

Missing production                            21
Zero production                             2185
Positive production                        19969
Negative production                            0
Positive target with zero production           0
Target greater than positive production        0
dtype: int64

In [22]:
missing_production_rows = labeled_candidates.loc[
    labeled_candidates["original_production"].isna(),
    [
        "source_row",
        "ID",
        "Month",
        "Previous_pdn",
        "Rej_and_cut_Piece",
        "original_production"
    ]
]

display(missing_production_rows)

,source_row,ID,Month,Previous_pdn,Rej_and_cut_Piece,original_production
86188,86188,13178-2,July,NaN,22.0,NaN
86627,86627,13178-2,July,NaN,22.0,NaN
86968,86968,13178-2,July,NaN,22.0,NaN
87356,87356,13178-2,July,NaN,22.0,NaN
87761,87761,13178-2,July,NaN,22.0,NaN
88173,88173,13178-2,July,NaN,22.0,NaN
88595,88595,13178-2,July,NaN,22.0,NaN
89025,89025,13178-2,July,NaN,22.0,NaN
89401,89401,13178-2,July,NaN,22.0,NaN
89788,89788,13178-2,July,NaN,22.0,NaN


In [23]:
order_records = labeled_candidates.loc[
    labeled_candidates["ID"].eq("13178-2"),
    [
        "Month",
        "Previous_pdn",
        "Rej_and_cut_Piece",
        "original_production"
    ]
]

# Show distinct combinations for inspection only
display(order_records.drop_duplicates())

,Month,Previous_pdn,Rej_and_cut_Piece,original_production
76036,June,TOTAL,0.0,0.0
77291,June,TOTAL,5.0,64.0
79264,June,TOTAL,5.0,418.0
80611,June,TOTAL,5.0,866.0
81981,June,TOTAL,5.0,1243.0
82472,July,TOTAL,5.0,1243.0
82908,July,TOTAL,5.0,1641.0
83811,July,TOTAL,10.0,1646.0
84278,July,TOTAL,10.0,2163.0
85748,July,TOTAL,10.0,2689.0


In [24]:
# Inspect the original rows around the first occurrence of 22
context = raw_excel.iloc[86185:86193].copy()
context.insert(0, "source_row", context.index)

with pd.option_context("display.max_columns", None):
    display(context)

,source_row,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21
86185,86185,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,2100.0,2296.560,0,1915,1915.0,16.614415,NaN,NaN,NaN,NaN,NaN,NaN,B/F,NaN
86186,86186,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6350.0,6944.360,NaN,TOTAL,5725.0,17.558998,NaN,5916.0,5916.0,74.0,5990.0,13.742951,NaN,NaN
86187,86187,13178-2,40x40/120x80,2956.0,7.0,3178.494624,12.5,3321.658087,73.0,3340.0,3652.624,154,2812,2966.0,18.798102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86188,86188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,NaN,NaN,NaN,NaN
86189,86189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3340.0,3652.624,NaN,TOTAL,2966.0,18.798102,NaN,2679.0,2679.0,12.0,2691.0,26.326936,NaN,NaN
86190,86190,13178-3,40x40/120x80,2780.0,7.0,2989.247312,12.5,3123.886834,135.0,3115.0,3406.564,189,2501,2690.0,21.034802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86191,86191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86192,86192,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3115.0,3406.564,NaN,TOTAL,2690.0,21.034802,NaN,2016.0,2016.0,83.0,2099.0,38.383662,NaN,NaN


In [25]:
# Keep ambiguous records separately; do not overwrite the original data
ambiguous_records = labeled_candidates.loc[
    labeled_candidates["original_production"].isna()
].copy()

# Working candidates with an explicitly recorded target
# and positive recorded production
production_candidates = labeled_candidates.loc[
    labeled_candidates["original_production"].gt(0)
].copy()

print("Ambiguous records set aside:", len(ambiguous_records))
print("Positive-production candidates:", len(production_candidates))
print("IDs represented:", production_candidates["ID"].nunique())
print(
    "Recorded zero targets retained:",
    production_candidates["Rej_and_cut_Piece"].eq(0).sum()
)

Ambiguous records set aside: 21
Positive-production candidates: 19969
IDs represented: 946
Recorded zero targets retained: 4563


In [26]:
# Compare the original dataset columns.
# Exclude source_row because each row position is unique.
comparison_columns = df.columns.tolist()

repeated = production_candidates.duplicated(
    subset=comparison_columns,
    keep="first"
)

print("Working rows:", len(production_candidates))
print("Exact repeats beyond the first:", repeated.sum())
print("Distinct full records:", (~repeated).sum())

print("\nNumber of records per ID:")
display(
    production_candidates.groupby("ID").size().describe()
)

Working rows: 19969
Exact repeats beyond the first: 10318
Distinct full records: 9651

Number of records per ID:


count    946.000000
mean      21.108879
std       12.079708
min        1.000000
25%       12.000000
50%       20.000000
75%       29.000000
max       93.000000
dtype: float64

In [27]:
# Save the working candidates without removing repeated records
production_candidates.to_csv(
    "../data/processed/production_candidates.csv",
    index=False
)

# Save a compact audit summary
audit_summary = pd.Series({
    "raw_rows": len(df),
    "raw_unique_ids": df["ID"].nunique(),
    "originally_missing_targets": numeric_target.isna().sum(),
    "recorded_target_rows": len(labeled_candidates),
    "positive_production_candidates": len(production_candidates),
    "candidate_unique_ids": production_candidates["ID"].nunique(),
    "exact_repeats_beyond_first": repeated.sum(),
    "distinct_full_records": (~repeated).sum()
}, name="count")

audit_summary.to_csv("../reports/dataset_audit_summary.csv")

print("Candidate data and audit summary saved.")

Candidate data and audit summary saved.
